# Taller I · Apache Spark: RDDs, DataFrames y Spark SQL

| | |
|---|---|
| **Curso** | MCDI502 · Gestión de Datos y Tecnologías Big Data |
| **Programa** | Magíster en Ciencia de Datos · Universidad Andrés Bello |
| **Evaluación** | Sumativa 2 · Taller I |
| **Grupo** | 6 |
| **Integrantes** | Fernanda Ovalle Román<br>Sebastián Cajales Cid<br>César Lorca Bacián<br>Jorge Álvarez Ossandón |
| **Repositorio** | [github.com/jaossandon/mcdi502-s2-spark](https://github.com/jaossandon/mcdi502-s2-spark) |

**Datos:** registros históricos de los Juegos Olímpicos (1896–2016) en seis archivos: deportistas (dos partes), equipos, eventos, juegos y resultados. La tabla de **resultados** es el punto de conexión que relaciona deportistas, equipos, juegos y eventos.

## Índice

| # | Sección | Indicador | Puntaje |
|---|---|---|---|
| 1 | Configuración inicial del entorno | – | 6 |
| 2 | RDDs: creación y unión | ID 3.1 | 18 |
| 3 | RDDs: transformaciones | ID 3.2 | 12 |
| 4 | DataFrames: creación e integración | ID 3.1 | 18 |
| 5 | Paralelismo | ID 3.4 | 9 |
| 6 | Inspección del DataFrame | ID 3.1 | 6 |
| 7 | Columnas calculadas | ID 3.3 | 12 |
| 8 | Agregaciones requeridas | ID 3.3 | 21 |
| 9 | Conclusiones y trazabilidad con la rúbrica | – | – |

### Cómo leer este notebook
Cada paso tiene tres partes:
1. **🎯 Qué haremos**: objetivo, decisión técnica y criterio de la rúbrica que cubre.
2. **Código**, comentado.
3. **✅ Resultado**: qué se obtuvo y cómo se interpreta.

---
# 1. Configuración inicial del entorno

Apache Spark está escrito en Scala y se ejecuta sobre la **JVM (Java Virtual Machine)**. Para usarlo desde Python se necesitan dos piezas: **Java**, que ejecuta el motor de Spark, y **PySpark**, la interfaz de Python que le envía las instrucciones.

## 1.1 Instalación de Java y Apache Spark

**🎯 Qué haremos**
- Detectar si el notebook se ejecuta en **Google Colab** o en un equipo local.
- En Colab: instalar **OpenJDK 17** y **PySpark 4.0.1** (el paquete ya incluye los binarios de Spark).
- En local: usar el entorno virtual del proyecto, donde ya están instalados.

**Decisión técnica:** se fijan versiones exactas (PySpark 4.0.1 y Java 17) para que el notebook sea **reproducible**. Spark 4.0 es la primera versión compatible con Python 3.13 y requiere Java 17 o superior.

In [1]:
# 1.1 Instalación de Java y Apache Spark
import sys, subprocess

EN_COLAB = "google.colab" in sys.modules
print("¿Ejecutando en Google Colab?:", EN_COLAB)

if EN_COLAB:
    # Java 17 sin interfaz gráfica (más liviano)
    subprocess.run("apt-get update -qq > /dev/null && "
                   "apt-get install -y -qq openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    # PySpark con la misma versión que el entorno local
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.1"], check=True)

# Evidencia: versiones disponibles
import pyspark
print("Versión de Python :", sys.version.split()[0])
print("Versión de PySpark:", pyspark.__version__)

¿Ejecutando en Google Colab?: False
Versión de Python : 3.11.9
Versión de PySpark: 4.0.1


**✅ Resultado**

El notebook detecta que corre en un equipo local (`False`), por lo que no reinstala nada y usa el entorno virtual del proyecto: **Python 3.13.12** y **PySpark 4.0.1**. En Colab, la misma celda instala Java 17 y PySpark 4.0.1 automáticamente, de modo que el notebook funciona en ambos entornos sin cambios.

## 1.2 Variables de entorno

**🎯 Qué haremos**

Definir las variables de entorno que Spark lee al arrancar:

| Variable | Para qué sirve |
|---|---|
| `JAVA_HOME` | Carpeta de Java que Spark usa para lanzar la JVM. |
| `SPARK_HOME` | Carpeta con los binarios de Spark (dentro del paquete `pyspark`). |
| `PYSPARK_PYTHON` | Intérprete de Python de los *workers* (procesos que ejecutan las tareas). |
| `PYSPARK_DRIVER_PYTHON` | Intérprete de Python del *driver* (proceso que coordina el trabajo). |

**Decisiones técnicas:**
- `JAVA_HOME` apunta **explícitamente a Java 17**. El equipo local también tiene Java 8, que no es compatible con Spark 4.0; sin esta variable, Spark podría tomar la versión equivocada.
- *Driver* y *workers* usan **el mismo intérprete de Python**. Si las versiones no coinciden, Spark falla al ejecutar funciones de Python sobre los RDDs.

In [2]:
# 1.2 Variables de entorno
import os, glob, platform

if EN_COLAB:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
elif platform.system() == "Windows":
    # Busca la instalación de Java 17 (Eclipse Temurin)
    jdks = sorted(glob.glob(r"C:\Program Files\Eclipse Adoptium\jdk-17*"))
    os.environ["JAVA_HOME"] = jdks[-1]

# Java 17 primero en el PATH, para que tenga prioridad sobre Java 8
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Evidencia: valores configurados
for var in ["JAVA_HOME", "SPARK_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON"]:
    print(f"{var:22s} = {os.environ[var]}")

# Evidencia: versión de Java que usará Spark
java = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
salida = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
print("\nJava que usará Spark:", [l for l in salida.splitlines() if "version" in l][0])

JAVA_HOME              = C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot
SPARK_HOME             = c:\Users\jorge\Documents\mcdi502-s2-spark\.venv\Lib\site-packages\pyspark
PYSPARK_PYTHON         = c:\Users\jorge\Documents\mcdi502-s2-spark\.venv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON  = c:\Users\jorge\Documents\mcdi502-s2-spark\.venv\Scripts\python.exe

Java que usará Spark: openjdk version "17.0.21" 2026-10-20


**✅ Resultado**

Las cuatro variables quedan definidas. `JAVA_HOME` apunta a **Java 17.0.21** (Eclipse Temurin) y no a la versión 8 que también existe en el equipo; la verificación final lo confirma (`openjdk version "17.0.21"`). `SPARK_HOME`, `PYSPARK_PYTHON` y `PYSPARK_DRIVER_PYTHON` apuntan al entorno virtual del proyecto, por lo que *driver* y *workers* usan el mismo intérprete.

## 1.3 Creación de la SparkSession y el SparkContext

**🎯 Qué haremos**

Crear los dos puntos de entrada de Spark:
- **`SparkSession` (`spark`)**: punto de entrada a **DataFrames y Spark SQL** (secciones 4 a 8).
- **`SparkContext` (`sc`)**: punto de entrada a los **RDDs** (secciones 2 y 3). Se obtiene con `spark.sparkContext`, de modo que ambos comparten la misma aplicación.

**Decisiones técnicas:**

| Configuración | Valor | Justificación |
|---|---|---|
| `master` | `local[*]` | Ejecuta Spark en este equipo usando **todos los núcleos** disponibles como *workers*. |
| `appName` | `MCDI502_S2_Grupo6` | Identifica la aplicación en la interfaz web de Spark. |
| `spark.sql.shuffle.partitions` | `8` | El valor por defecto (200) está pensado para clústeres. Con datos de este tamaño generaría cientos de tareas casi vacías tras cada `join` o `groupBy`. |
| `setLogLevel("ERROR")` | – | Oculta avisos internos de Spark para que las salidas se lean con claridad. |

In [3]:
# 1.3 SparkSession y SparkContext
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("MCDI502_S2_Grupo6")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

sc = spark.sparkContext          # SparkContext: punto de entrada a los RDDs
sc.setLogLevel("ERROR")

# Evidencia: la sesión está activa
print("Versión de Spark        :", spark.version)
print("Nombre de la aplicación :", sc.appName)
print("Master                  :", sc.master)
print("Paralelismo por defecto :", sc.defaultParallelism, "núcleos")
print("Particiones de shuffle  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Interfaz web de Spark   :", sc.uiWebUrl)

Versión de Spark        : 4.0.1
Nombre de la aplicación : MCDI502_S2_Grupo6
Master                  : local[*]
Paralelismo por defecto : 8 núcleos
Particiones de shuffle  : 8
Interfaz web de Spark   : http://DESKTOP-KD98F3J:4040


**✅ Resultado**

La sesión queda activa con **Spark 4.0.1** en modo `local[*]`. El paralelismo por defecto es **8**, igual al número de núcleos del equipo: Spark puede ejecutar hasta 8 tareas al mismo tiempo, una por núcleo. Las particiones de *shuffle* quedan en 8, alineadas con ese paralelismo. Desde aquí, `spark` se usa para DataFrames y SQL, y `sc` para RDDs. La interfaz web en el puerto 4040 permite monitorear los trabajos mientras se ejecutan.

## 1.4 Ubicación de los datos

**🎯 Qué haremos**

Dejar disponibles los seis archivos originales en una carpeta `data/`:
- **En local:** se usa la carpeta `data/` del repositorio, donde ya están los archivos.
- **En Colab (u otro equipo):** la carpeta no existe, así que se descargan desde el repositorio público del grupo en GitHub.

**Decisión técnica:** la entrega incluye solo el notebook y el video, no los datos. Si el notebook dependiera de subir archivos a mano, no podría ejecutarse "sin intervenciones manuales". Descargarlos desde GitHub hace que cualquier persona pueda ejecutarlo de principio a fin. La ruta se guarda en formato `/` (`as_posix`), que Spark entiende tanto en Windows como en Linux.

In [4]:
# 1.4 Ubicación de los datos
import urllib.request
from pathlib import Path

URL_BASE = "https://raw.githubusercontent.com/jaossandon/mcdi502-s2-spark/main/data/"
ARCHIVOS = ["deportista.csv", "deportista2.csv", "equipo.csv",
            "evento.csv", "resultados.csv", "juegos.json"]

# Busca data/ en la carpeta actual o en las superiores (el notebook está en notebooks/)
carpetas = [Path.cwd(), *Path.cwd().parents]
DATA_DIR = next((p / "data" for p in carpetas if (p / "data" / "deportista.csv").exists()),
                Path("data"))
DATA_DIR.mkdir(exist_ok=True)

for nombre in ARCHIVOS:
    destino = DATA_DIR / nombre
    if not destino.exists():                       # solo descarga si falta
        urllib.request.urlretrieve(URL_BASE + nombre, destino)

DATA = DATA_DIR.resolve().as_posix()               # ruta que usará Spark

# Evidencia: archivos disponibles
print("Carpeta de datos:", DATA, "\n")
print(f"{'Archivo':18s}{'Tamaño (bytes)':>16s}")
for nombre in ARCHIVOS:
    print(f"{nombre:18s}{(DATA_DIR / nombre).stat().st_size:>16,d}")

Carpeta de datos: C:/Users/jorge/Documents/mcdi502-s2-spark/data 

Archivo             Tamaño (bytes)
deportista.csv           2,832,265
deportista2.csv          2,874,875
equipo.csv                  23,606
evento.csv                  35,144
resultados.csv           6,443,911
juegos.json                  5,232


**✅ Resultado**

Los seis archivos están disponibles en la carpeta `data/` del repositorio:

| Archivo | Tamaño | Contenido |
|---|---|---|
| `deportista.csv` | 2,8 MB | deportistas con id 1 al 67.786 |
| `deportista2.csv` | 2,9 MB | deportistas con id 67.787 al 135.571 |
| `equipo.csv` | 24 KB | 1.184 equipos |
| `evento.csv` | 35 KB | 765 eventos |
| `resultados.csv` | 6,4 MB | 271.116 participaciones |
| `juegos.json` | 5 KB | 51 ediciones de los Juegos |

**Cierre de la sección 1:** Java 17 y PySpark 4.0.1 operativos, variables de entorno definidas, `SparkSession` y `SparkContext` activos y datos disponibles. Todo se ejecuta sin intervención manual, tanto en local como en Colab.

## 2. RDDs: creación y unión (ID 3.1 · 18 pts)

Un **RDD** (*Resilient Distributed Dataset*) es la estructura básica de Spark: una colección de elementos **inmutable**, **distribuida en particiones** y **tolerante a fallos** (si una partición se pierde, Spark la reconstruye con su linaje de transformaciones).

### 2.1 RDD `deportista` con 6 particiones

🎯 **Qué haremos**
- Leer `deportista.csv` como texto con `sc.textFile()`. Cada elemento del RDD será **una línea** del archivo.
- Pedir **6 particiones** con `minPartitions=6`, tal como exige la pauta. Cada partición es un bloque de datos que un núcleo procesa en paralelo.
- Verificar el número de particiones con `getNumPartitions()` y mirar las primeras líneas con `take(3)`.

**Decisión técnica:** el archivo **no trae encabezado**, así que no hay que eliminar una fila de títulos antes de unir y contar.

In [5]:
# 2.1 Crear el RDD deportista con 6 particiones
deportista = sc.textFile(f"{DATA}/deportista.csv", minPartitions=6)

print("Tipo de objeto          :", type(deportista).__name__)
print("Número de particiones   :", deportista.getNumPartitions())
print("Primeras 3 líneas (texto):")
for linea in deportista.take(3):
    print("  ", linea)

Tipo de objeto          : RDD
Número de particiones   : 6
Primeras 3 líneas (texto):
   1,A Dijiang,1,24,180,80,199
   2,A Lamusi,1,23,170,60,199
   3,Gunnar Nielsen Aaby,1,24,0,0,273


✅ **Resultado**
- Se creó el RDD `deportista` con **6 particiones**, como pide la pauta (`getNumPartitions() = 6`).
- Cada elemento es **una línea de texto** (`'1,A Dijiang,1,24,180,80,199'`): el RDD todavía no conoce columnas ni tipos. Eso lo resolveremos al convertirlo a DataFrame.
- `textFile` es una operación **perezosa (lazy)**: Spark solo leyó el archivo cuando ejecutamos la **acción** `take(3)`.
- En la tercera línea aparecen `0,0` en altura y peso. Un deportista no puede medir 0 cm, así que trataremos esos ceros como **valores faltantes (null)**.

In [6]:
# 2.2 RDD deportista2, unión y conteo
deportista2 = sc.textFile(f"{DATA}/deportista2.csv")

deportistaTotal = deportista.union(deportista2)

n1, n2, nt = deportista.count(), deportista2.count(), deportistaTotal.count()
print(f"{'RDD':16s} {'Registros':>10s} {'Particiones':>12s}")
print(f"{'deportista':16s} {n1:>10,d} {deportista.getNumPartitions():>12d}")
print(f"{'deportista2':16s} {n2:>10,d} {deportista2.getNumPartitions():>12d}")
print(f"{'deportistaTotal':16s} {nt:>10,d} {deportistaTotal.getNumPartitions():>12d}")
print("\n¿La unión conserva todos los registros?:", nt == n1 + n2)
print("Primera línea de deportista2:", deportista2.first())

RDD               Registros  Particiones
deportista           67,786            6
deportista2          67,785            2
deportistaTotal     135,571            8

¿La unión conserva todos los registros?: True
Primera línea de deportista2: 67787,Lee BongJu,1,27,167,56,970


✅ **Resultado**

| RDD | Registros | Particiones |
|---|---:|---:|
| `deportista` | 67.786 | 6 |
| `deportista2` | 67.785 | 2 |
| `deportistaTotal` | **135.571** | **8** |

- La unión conserva **todos** los registros: 67.786 + 67.785 = 135.571 (`True`). No se pierde ni se duplica ninguna fila.
- `deportista2` empieza en el id **67.787**, justo donde termina `deportista` (67.786). Los archivos son **dos mitades contiguas** de la misma tabla, sin ids repetidos.
- `deportistaTotal` tiene **6 + 2 = 8 particiones**: `union()` solo concatena las particiones, sin *shuffle*, lo que la hace una operación barata.
- `deportista2` quedó con 2 particiones porque no le pedimos un número; Spark usa por defecto `min(núcleos, 2)`.

### 2.3 Parseo del RDD: de líneas de texto a registros con tipo

🎯 **Qué haremos**
- Definir una función `parsear()` que separa cada línea por comas y convierte cada campo a su tipo: `deportista_id`, `genero`, `edad` y `equipo_id` a **entero**; `altura` y `peso` a **decimal**; `nombre` como texto.
- Aplicarla a todo `deportistaTotal` con la transformación `map()`, obteniendo el RDD `deportistaRDD` de **tuplas**.

**Decisiones técnicas**
1. **Ceros como nulos:** en `edad`, `altura` y `peso` un valor `0` significa "dato no registrado" (nadie mide 0 cm). Los convertimos a `None` para que no distorsionen promedios ni mínimos.
2. **Cortar a 7 campos (`[:7]`):** la fila del deportista **16587** trae una coma extra al final, lo que genera una 8.ª columna vacía. Tomar solo los primeros 7 campos lo resuelve sin perder información.

In [7]:
# 2.3 Parsear cada línea a una tupla con tipos
def a_nulo(valor, tipo):
    """Convierte el texto al tipo indicado; '0' o vacío se consideran dato faltante (None)."""
    valor = valor.strip()
    return None if valor in ("", "0") else tipo(valor)

def parsear(linea):
    c = linea.split(",")[:7]          # solo 7 campos (la fila 16587 trae una coma extra)
    return (int(c[0]),                # deportista_id
            c[1].strip(),             # nombre
            int(c[2]),                # genero (1 = hombre, 2 = mujer)
            a_nulo(c[3], int),        # edad
            a_nulo(c[4], float),      # altura (cm)
            a_nulo(c[5], float),      # peso (kg)
            int(c[6]))                # equipo_id

deportistaRDD = deportistaTotal.map(parsear)

print("Fila 16587 en bruto :", deportistaTotal.filter(lambda l: l.startswith("16587,")).first())
print("Fila 16587 parseada :", deportistaRDD.filter(lambda r: r[0] == 16587).first())
print("\nPrimeras 3 tuplas:")
for t in deportistaRDD.take(3):
    print("  ", t)
print("\nRegistros parseados:", f"{deportistaRDD.count():,}")

Fila 16587 en bruto : 16587,Vladimr Bil,1,24,191,92,67,
Fila 16587 parseada : (16587, 'Vladimr Bil', 1, 24, 191.0, 92.0, 67)

Primeras 3 tuplas:
   (1, 'A Dijiang', 1, 24, 180.0, 80.0, 199)
   (2, 'A Lamusi', 1, 23, 170.0, 60.0, 199)
   (3, 'Gunnar Nielsen Aaby', 1, 24, None, None, 273)

Registros parseados: 135,571


✅ **Resultado**
- Las **135.571** líneas se parsearon sin errores a tuplas de 7 campos con sus tipos (`int`, `str`, `float`).
- **Fila 16587:** en bruto termina en una coma extra (`...,92,67,`). Tras el corte `[:7]` queda con sus 7 valores correctos (`equipo_id = 67`), sin perder datos.
- **Ceros a nulos:** el deportista 3 (*Gunnar Nielsen Aaby*) tenía `0,0` en altura y peso, y ahora tiene `None, None`. Así esos ceros no bajarán artificialmente los promedios ni aparecerán como mínimos.
- `map()` es una **transformación**: no se ejecutó nada hasta las acciones `first()`, `take()` y `count()`.

### 2.4 Conversión del RDD a DataFrame `deportista`

🎯 **Qué haremos**
- Definir un **esquema explícito** con `StructType` que nombra las 7 columnas y fija su tipo.
- Crear el DataFrame `deportista` con `spark.createDataFrame(deportistaRDD, esquema)`.
- Mostrar las primeras filas, el esquema y la cantidad de **nulos por columna**, para documentar el efecto de la decisión "ceros como nulos".

**Decisión técnica:** con un esquema explícito, Spark **no infiere tipos** (eso lo obligaría a recorrer los datos) y garantiza que `edad` sea entera y `altura`/`peso` decimales. Usamos el nombre `deportista` que pide la pauta. Desde aquí, `deportista` pasa a ser el **DataFrame**; el RDD original sigue disponible dentro de `deportistaTotal`.

In [8]:
# 2.4 Convertir el RDD a DataFrame con esquema explícito
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

esquema_deportista = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre",        StringType(),  True),
    StructField("genero",        IntegerType(), True),
    StructField("edad",          IntegerType(), True),
    StructField("altura",        DoubleType(),  True),
    StructField("peso",          DoubleType(),  True),
    StructField("equipo_id",     IntegerType(), True),
])

deportista = spark.createDataFrame(deportistaRDD, esquema_deportista)

deportista.show(5)
deportista.printSchema()
print(f"Filas del DataFrame deportista: {deportista.count():,}")

print("\nNulos por columna:")
deportista.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                   for c in deportista.columns]).show()

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24| 180.0|80.0|      199|
|            2|            A Lamusi|     1|  23| 170.0|60.0|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|  NULL|NULL|      273|
|            4|Edgar Lindenau Aabye|     1|  34|  NULL|NULL|      278|
|            5|Christine Jacoba ...|     2|  21| 185.0|82.0|      705|
+-------------+--------------------+------+----+------+----+---------+
only showing top 5 rows
root
 |-- deportista_id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)

Filas del DataFrame deportista: 135,571

Nulos por col

✅ **Resultado**
- Se creó el **DataFrame `deportista`** con las 7 columnas que pide la pauta y **135.571 filas**, las mismas del RDD `deportistaTotal`.
- El esquema confirma los tipos: `deportista_id`, `genero`, `edad` y `equipo_id` como **integer**; `altura` y `peso` como **double**; `nombre` como **string**.
- **Nulos por columna** (efecto de tratar los ceros como dato faltante):

| Columna | Nulos | % del total |
|---|---:|---:|
| edad | 6.368 | 4,7 % |
| altura | 33.916 | 25,0 % |
| peso | 34.885 | 25,7 % |
| resto | 0 | 0 % |

- Uno de cada cuatro deportistas no tiene altura ni peso registrados, sobre todo en los Juegos más antiguos. Si los hubiéramos dejado como `0`, el promedio de altura habría caído artificialmente y el mínimo sería 0 cm. Esto será importante en el **IMC** (sección 7) y en las **agregaciones** (sección 8).

**Cierre de la sección 2:** creamos el RDD `deportista` con **6 particiones**, el RDD `deportista2`, los unimos en `deportistaTotal` (**135.571** registros, 8 particiones) y lo convertimos al DataFrame `deportista` con un esquema explícito.

## 3. RDDs: transformaciones (ID 3.2 · 12 pts)

Las **transformaciones** (`map`, `filter`, ...) crean un RDD nuevo a partir de otro, sin modificar el original (los RDDs son inmutables), y son **perezosas**: solo se ejecutan cuando llega una **acción** (`count`, `take`, `collect`...).

### 3.1 RDD `MayorEdad`: deportistas de 18 años o más

🎯 **Qué haremos**
- Aplicar `filter()` sobre `deportistaRDD` (las tuplas tipadas de la sección 2) para quedarnos con los deportistas con **edad ≥ 18**.
- Mostrar el **efecto** del filtro: registros antes y después, y verificar que la edad mínima del nuevo RDD sea 18.

**Decisión técnica:** filtramos sobre `deportistaRDD` y no sobre el texto, porque la edad ya es un **entero** (comparar `"9" >= "18"` como texto daría un resultado incorrecto). Los deportistas **sin edad registrada** (`None`) quedan fuera: no sabemos si son mayores de edad.

In [9]:
# 3.1 RDD MayorEdad: edad >= 18
EDAD = 3   # posición de la edad en la tupla

MayorEdad = deportistaRDD.filter(lambda r: r[EDAD] is not None and r[EDAD] >= 18)

total     = deportistaRDD.count()
mayores   = MayorEdad.count()
menores   = deportistaRDD.filter(lambda r: r[EDAD] is not None and r[EDAD] < 18).count()
sin_edad  = deportistaRDD.filter(lambda r: r[EDAD] is None).count()

print(f"Deportistas en total      : {total:>9,d}")
print(f"  MayorEdad (edad >= 18)  : {mayores:>9,d}  ({mayores/total:.1%})")
print(f"  Menores de 18           : {menores:>9,d}")
print(f"  Sin edad registrada     : {sin_edad:>9,d}")
print(f"  ¿Suman el total?        : {mayores + menores + sin_edad == total}")
print(f"\nEdad mínima en MayorEdad  : {MayorEdad.map(lambda r: r[EDAD]).min()}")
print(f"Edad máxima en MayorEdad  : {MayorEdad.map(lambda r: r[EDAD]).max()}")
print("\nPrimeros 3 registros de MayorEdad:")
for r in MayorEdad.take(3):
    print("  ", r)

Deportistas en total      :   135,571
  MayorEdad (edad >= 18)  :   123,244  (90.9%)
  Menores de 18           :     5,959
  Sin edad registrada     :     6,368
  ¿Suman el total?        : True

Edad mínima en MayorEdad  : 18
Edad máxima en MayorEdad  : 97

Primeros 3 registros de MayorEdad:
   (1, 'A Dijiang', 1, 24, 180.0, 80.0, 199)
   (2, 'A Lamusi', 1, 23, 170.0, 60.0, 199)
   (3, 'Gunnar Nielsen Aaby', 1, 24, None, None, 273)


✅ **Resultado**
- `MayorEdad` contiene **123.244** deportistas (**90,9 %** del total), todos con edad entre **18 y 97** años. La edad mínima es 18, así que el filtro funcionó.
- El total se descompone sin pérdidas: **123.244** mayores + **5.959** menores + **6.368** sin edad = **135.571** (`True`).
- Los 6.368 registros sin edad quedan fuera del filtro, porque no podemos afirmar que sean mayores de edad. Esa es la consecuencia directa de haber tratado los ceros como nulos.
- La edad máxima de 97 años parece un error, pero corresponde a competidores de las antiguas competencias de **arte** olímpicas, que se otorgaban por obras y no por desempeño físico.

### 3.2 RDD `Deportistas_mujer`: deportistas con `genero = 2`

🎯 **Qué haremos**
- Filtrar `deportistaRDD` con `filter()` para quedarnos con las deportistas de **género 2 (mujer)**.
- Mostrar el **efecto**: distribución de género antes del filtro (con `map` + `reduceByKey`, el patrón clave-valor típico de RDDs) y comprobar que después del filtro **solo queda el género 2**.

**Decisión técnica:** `reduceByKey` suma los conteos **dentro de cada partición antes de mover datos** entre ellas (combinación local), por lo que es más eficiente que agrupar todos los registros y contarlos después (`groupByKey`).

In [10]:
# 3.2 RDD Deportistas_mujer: genero == 2
GENERO = 2   # posición del género en la tupla

Deportistas_mujer = deportistaRDD.filter(lambda r: r[GENERO] == 2)

# Distribución de género ANTES del filtro (patrón clave-valor)
dist_genero = (deportistaRDD.map(lambda r: (r[GENERO], 1))
                            .reduceByKey(lambda a, b: a + b)
                            .sortByKey()
                            .collect())
print("Distribución de género antes del filtro (genero, cantidad):", dist_genero)

n_mujeres = Deportistas_mujer.count()
print(f"\nDeportistas_mujer          : {n_mujeres:,} ({n_mujeres/deportistaRDD.count():.1%} del total)")
print("Géneros presentes después  :", Deportistas_mujer.map(lambda r: r[GENERO]).distinct().collect())
print("\nPrimeros 3 registros de Deportistas_mujer:")
for r in Deportistas_mujer.take(3):
    print("  ", r)

Distribución de género antes del filtro (genero, cantidad): [(1, 101590), (2, 33981)]

Deportistas_mujer          : 33,981 (25.1% del total)
Géneros presentes después  : [2]

Primeros 3 registros de Deportistas_mujer:
   (5, 'Christine Jacoba Aaftink', 2, 21, 185.0, 82.0, 705)
   (8, 'Cornelia Cor Aalten Strannood', 2, 18, 168.0, None, 705)
   (13, 'Minna Maarit Aalto', 2, 30, 159.0, 55.5, 350)


✅ **Resultado**
- Antes del filtro hay **101.590 hombres** (género 1) y **33.981 mujeres** (género 2), que suman los 135.571 deportistas.
- `Deportistas_mujer` contiene **33.981** registros (**25,1 %** del total). Después del filtro el único género presente es `[2]`, así que el filtro es correcto.
- La proporción refleja la **historia de los Juegos**: las mujeres no compitieron en 1896 y su participación creció gradualmente durante el siglo XX.
- Los nulos de altura y peso **no siempre coinciden**: *Cornelia Cor Aalten Strannood* tiene altura (168) pero no peso. Lo tendremos en cuenta al calcular el IMC, que necesita ambos datos.

### 3.3 Transformación a mayúsculas de `deportistaTotal`

🎯 **Qué haremos**
- Aplicar `map()` con `str.upper()` sobre **cada línea** de `deportistaTotal` para pasar todo su contenido a **mayúsculas**, en el nuevo RDD `deportistaMayus`.
- Mostrar el **efecto**: las mismas líneas antes y después, verificar que la cantidad de registros no cambia y que no queda **ninguna letra minúscula**.

**Decisión técnica:** usamos `map()` porque es una transformación **1 a 1** (cada línea produce exactamente una línea), sin *shuffle* y conservando las mismas 8 particiones. Trabajamos sobre `deportistaTotal` (el texto), como pide la pauta, ya que solo los nombres tienen letras; los números no cambian.

In [11]:
# 3.3 Pasar a mayúsculas todo el RDD deportistaTotal
deportistaMayus = deportistaTotal.map(lambda linea: linea.upper())

print("Antes (deportistaTotal)            ->  Después (deportistaMayus)")
for antes, despues in zip(deportistaTotal.take(4), deportistaMayus.take(4)):
    print(f"  {antes:34s} ->  {despues}")

print(f"\nRegistros antes / después : {deportistaTotal.count():,} / {deportistaMayus.count():,}")
print(f"Particiones antes/después : {deportistaTotal.getNumPartitions()} / {deportistaMayus.getNumPartitions()}")
print(f"Líneas con minúsculas antes  : {deportistaTotal.filter(lambda l: l != l.upper()).count():,}")
print(f"Líneas con minúsculas después: {deportistaMayus.filter(lambda l: l != l.upper()).count():,}")

Antes (deportistaTotal)            ->  Después (deportistaMayus)
  1,A Dijiang,1,24,180,80,199        ->  1,A DIJIANG,1,24,180,80,199
  2,A Lamusi,1,23,170,60,199         ->  2,A LAMUSI,1,23,170,60,199
  3,Gunnar Nielsen Aaby,1,24,0,0,273 ->  3,GUNNAR NIELSEN AABY,1,24,0,0,273
  4,Edgar Lindenau Aabye,1,34,0,0,278 ->  4,EDGAR LINDENAU AABYE,1,34,0,0,278

Registros antes / después : 135,571 / 135,571
Particiones antes/después : 8 / 8
Líneas con minúsculas antes  : 135,571
Líneas con minúsculas después: 0


✅ **Resultado**
- Todas las líneas pasaron a mayúsculas: antes **135.571** líneas tenían minúsculas (todas incluyen un nombre) y después **0**.
- La cantidad de registros (**135.571**) y de particiones (**8**) no cambia: `map()` es una transformación 1 a 1 y **sin *shuffle***.
- Solo cambian los nombres (`A Dijiang` → `A DIJIANG`); los valores numéricos quedan intactos.
- `deportistaTotal` **no se modificó**: `map()` creó un RDD nuevo (`deportistaMayus`), porque los RDDs son **inmutables**.

**Cierre de la sección 3**

| RDD | Transformación | Registros | Efecto verificado |
|---|---|---:|---|
| `MayorEdad` | `filter(edad ≥ 18)` | 123.244 | edad mínima = 18 |
| `Deportistas_mujer` | `filter(genero == 2)` | 33.981 | único género presente = 2 |
| `deportistaMayus` | `map(upper)` | 135.571 | 0 líneas con minúsculas |

Las tres son **transformaciones perezosas**: Spark solo construye el plan (linaje) y lo ejecuta cuando llega una **acción** como `count()` o `take()`.

## 4. DataFrames: creación, optimización e integración (ID 3.1 · 18 pts)

En esta sección creamos un DataFrame por cada tabla y luego los integramos en uno solo. Las **optimizaciones** aplicadas son:
1. **Esquemas explícitos con tipos correctos** (enteros para ids): Spark no tiene que inferir tipos y los joins comparan números, no textos.
2. **Nombres de columnas homogéneos** (`evento_id`, `equipo_id`, `juego_id`) para que los joins sean directos.
3. **`broadcast`** de las tablas pequeñas al integrar (se explica en 4.5).
4. **`cache`** del DataFrame integrado, que se reutiliza en todas las secciones siguientes.

### 4.1 DataFrame `evento`

🎯 **Qué haremos**
- Leer `evento.csv` como **texto** (RDD), quitar el encabezado y parsear cada línea con el módulo `csv` de Python.
- Convertirlo a DataFrame con esquema `evento_id INT, evento STRING, deporte_id INT`.

**Decisión técnica:** 280 de las 765 líneas vienen **envueltas completas entre comillas** (`"40,""Speed Skating Men's 1,500 metres"",5"`). Un lector CSV estándar las tomaría como **una sola columna**. Por eso quitamos las comillas externas, cambiamos `""` por `"` y recién entonces separamos los campos. Así también respetamos las comas **dentro** del nombre del evento (`1,500 metres`). Además, 2 eventos tienen `deporte_id = "#N/A"`, que convertimos a nulo.

In [12]:
# 4.1 DataFrame evento (lectura como texto + parseo manual)
import csv

def parsear_evento(linea):
    l = linea.strip()
    if l.startswith('"') and l.endswith('"'):      # línea envuelta completa en comillas
        l = l[1:-1].replace('""', '"')
    c = next(csv.reader([l]))                       # respeta comas dentro de comillas
    return (int(c[0]), c[1], None if c[2] == "#N/A" else int(c[2]))

evento_txt = sc.textFile(f"{DATA}/evento.csv")
encabezado = evento_txt.first()
print("Encabezado original       :", encabezado)
print("Líneas envueltas en comillas:", evento_txt.filter(lambda l: l.startswith('"')).count())

evento = spark.createDataFrame(
    evento_txt.filter(lambda l: l != encabezado).map(parsear_evento),
    "evento_id INT, evento STRING, deporte_id INT")

print(f"Filas del DataFrame evento: {evento.count():,}\n")
evento.show(5, truncate=False)
print("Casos especiales (comillas, comas internas y #N/A):")
evento.filter("evento_id IN (30, 40) OR deporte_id IS NULL").show(truncate=False)
evento.printSchema()

Encabezado original       : cod_evento,evento,deporte_id
Líneas envueltas en comillas: 280
Filas del DataFrame evento: 765

+---------+--------------------------------+----------+
|evento_id|evento                          |deporte_id|
+---------+--------------------------------+----------+
|1        |Basketball Men's Basketball     |1         |
|2        |Judo Men's Extra-Lightweight    |2         |
|3        |Football Men's Football         |3         |
|4        |Tug-Of-War Men's Tug-Of-War     |4         |
|5        |Speed Skating Women's 500 metres|5         |
+---------+--------------------------------+----------+
only showing top 5 rows
Casos especiales (comillas, comas internas y #N/A):
+---------+-------------------------------------------------+----------+
|evento_id|evento                                           |deporte_id|
+---------+-------------------------------------------------+----------+
|30       |Art Competitions Mixed Sculpturing, Unknown Event|14        |
|40 

✅ **Resultado**
- El DataFrame `evento` tiene **765 filas**, una por cada línea de datos del archivo, sin perder ninguna.
- Las **280 líneas envueltas en comillas** se parsearon correctamente: el evento 40 conserva la coma de `1,500 metres` y el 30 la de `Sculpturing, Unknown Event`, sin desplazar columnas.
- Los 2 eventos con `deporte_id = "#N/A"` (703 *Sailing* y 743 *Motorboating*) quedaron como `NULL`, en vez de romper la conversión a entero.
- Renombramos `cod_evento` → **`evento_id`**, el mismo nombre que usa la tabla de resultados, para facilitar el join.

### 4.2 DataFrame `equipo`

🎯 **Qué haremos**
- Leer `equipo.csv` con `spark.read.csv`, con encabezado y **esquema explícito** `equipo_id INT, equipo STRING, sigla STRING`.
- Mostrar las primeras filas, el conteo y verificar un caso con comillas internas.

**Decisiones técnicas**
- **Esquema explícito en lugar de `inferSchema`:** evita una pasada extra sobre el archivo para adivinar tipos y renombra `id` → **`equipo_id`**, el nombre que usa la tabla de deportistas.
- **`escape='"'`:** algunos nombres contienen comillas escritas como `""` (estándar CSV). Spark usa `\` como escape por defecto, así que sin esta opción el nombre quedaría como `"Large boat; ... ""Poros""-1"`.

In [13]:
# 4.2 DataFrame equipo
equipo = spark.read.csv(f"{DATA}/equipo.csv", header=True, escape='"',
                        schema="equipo_id INT, equipo STRING, sigla STRING")

print(f"Filas del DataFrame equipo: {equipo.count():,}")
print(f"Siglas (países/comités) distintas: {equipo.select('sigla').distinct().count()}\n")
equipo.show(5)
print("Caso con comillas internas y punto y coma en el nombre:")
equipo.filter("equipo_id IN (79, 565)").show(truncate=False)
equipo.printSchema()

Filas del DataFrame equipo: 1,184
Siglas (países/comités) distintas: 230

+---------+--------------------+-----+
|equipo_id|              equipo|sigla|
+---------+--------------------+-----+
|        1|         30. Februar|  AUT|
|        2|A North American ...|  MEX|
|        3|           Acipactli|  MEX|
|        4|             Acturus|  ARG|
|        5|         Afghanistan|  AFG|
+---------+--------------------+-----+
only showing top 5 rows
Caso con comillas internas y punto y coma en el nombre:
+---------+-----------------------------------------------+-----+
|equipo_id|equipo                                         |sigla|
+---------+-----------------------------------------------+-----+
|79       |Bagatelle Polo Club; Paris                     |FRA  |
|565      |Large boat; Central Naval Prep School "Poros"-1|GRE  |
+---------+-----------------------------------------------+-----+

root
 |-- equipo_id: integer (nullable = true)
 |-- equipo: string (nullable = true)
 |-- sigla: s

✅ **Resultado**
- El DataFrame `equipo` tiene **1.184 equipos** que representan a **230 siglas** (países o comités olímpicos). Un mismo país puede tener varios equipos, por ejemplo clubes o botes en los primeros Juegos.
- Con `escape='"'`, el equipo 565 queda como `Large boat; Central Naval Prep School "Poros"-1`, sin comillas duplicadas. El `;` dentro del nombre (equipo 79) tampoco causa problemas porque el separador es la coma.
- Los tipos son los correctos (`equipo_id` entero) y la columna se renombró de `id` a **`equipo_id`** para el join con `deportista`.

### 4.3 DataFrame `resultado`

🎯 **Qué haremos**
- Leer `resultados.csv` con separador **`;`**, encabezado y esquema explícito.
- Limpiar dos columnas:
  - `medalla`: el texto `"NA"` (sin medalla) pasa a **nulo** real.
  - `evento_id`: se lee como texto y se convierte a entero con **`try_cast`**, porque 9 filas traen `"#N/A"`.
- Mostrar el conteo, la distribución de medallas y el esquema final.

**Decisiones técnicas**
- **`"NA"` → nulo:** si la dejáramos como texto, `"NA"` aparecería como una "cuarta medalla" en las agregaciones, y `count("medalla")` contaría filas sin medalla.
- **`try_cast` en vez de `cast`:** Spark 4 trabaja en **modo ANSI** (estricto), donde convertir `"#N/A"` a entero produce un error. `try_cast` devuelve nulo en esos casos y mantiene el resto de los valores.

In [14]:
# 4.3 DataFrame resultado
resultado_raw = spark.read.csv(f"{DATA}/resultados.csv", sep=";", header=True,
    schema="resultado_id INT, medalla STRING, deportista_id INT, juego_id INT, evento_id STRING")

print("Antes de limpiar:")
print("  medalla = 'NA'     :", f"{resultado_raw.filter(F.col('medalla') == 'NA').count():,}")
print("  evento_id = '#N/A' :", resultado_raw.filter(F.col("evento_id") == "#N/A").count())

resultado = (resultado_raw
    .withColumn("medalla", F.when(F.col("medalla") == "NA", None).otherwise(F.col("medalla")))
    .withColumn("evento_id", F.expr("try_cast(evento_id AS INT)")))

print(f"\nFilas del DataFrame resultado: {resultado.count():,}")
print("evento_id nulos tras try_cast:", resultado.filter(F.col("evento_id").isNull()).count(), "\n")
resultado.show(5)
print("Distribución de medallas (NULL = participó sin medalla):")
resultado.groupBy("medalla").count().orderBy(F.desc("count")).show()
resultado.printSchema()

Antes de limpiar:
  medalla = 'NA'     : 231,333
  evento_id = '#N/A' : 9

Filas del DataFrame resultado: 271,116
evento_id nulos tras try_cast: 9 

+------------+-------+-------------+--------+---------+
|resultado_id|medalla|deportista_id|juego_id|evento_id|
+------------+-------+-------------+--------+---------+
|           1|   NULL|            1|      39|        1|
|           2|   NULL|            2|      49|        2|
|           3|   NULL|            3|       7|        3|
|           4|   Gold|            4|       2|        4|
|           5|   NULL|            5|      36|        5|
+------------+-------+-------------+--------+---------+
only showing top 5 rows
Distribución de medallas (NULL = participó sin medalla):
+-------+------+
|medalla| count|
+-------+------+
|   NULL|231333|
|   Gold| 13372|
| Bronze| 13295|
| Silver| 13116|
+-------+------+

root
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- deportista_id: integer (nullable = 

✅ **Resultado**
- El DataFrame `resultado` tiene **271.116 filas**: cada una es la participación de un deportista en un evento de unos Juegos.
- **Medallas:** 231.333 participaciones (**85,3 %**) no obtuvieron medalla y ahora son `NULL` real, no el texto `"NA"`. Las medallas se reparten casi en partes iguales: **13.372 oros**, **13.295 bronces** y **13.116 platas** (39.783 en total).
- Los **9** `evento_id = "#N/A"` quedaron como `NULL` gracias a `try_cast`, sin detener la ejecución por el modo ANSI. El resto de los ids son enteros válidos.
- En el esquema final todos los ids son **integer**, el tipo necesario para los joins con `deportista`, `evento` y `juego`.

### 4.4 DataFrame `juego`

🎯 **Qué haremos**
- Leer `juegos.json` con `spark.read.json` y la opción **`multiLine=True`**.
- Seleccionar y **renombrar** las columnas según su contenido real, con los tipos correctos.

**Decisiones técnicas**
- **`multiLine=True`:** el archivo es un único arreglo JSON repartido en varias líneas. Sin esta opción, Spark lo leería como registros corruptos (`_corrupt_record`).
- **Renombrar columnas:** los nombres originales no corresponden a su contenido:

| Columna original | Contenido real | Nuevo nombre |
|---|---|---|
| `ano` | `"1896 Verano"` | `nombre_juego` |
| `temporada` | `1896` | `anio` |
| `ciudad` | `"Verano"` / `"Invierno"` | `temporada` |

- **Tipos:** Spark infiere los números JSON como `long` (64 bits). Los convertimos a `int`, suficiente para ids y años, para que coincidan con `juego_id` de `resultado` y ocupen menos memoria.

In [15]:
# 4.4 DataFrame juego
juego_raw = spark.read.option("multiLine", True).json(f"{DATA}/juegos.json")
print("Esquema original (inferido del JSON):")
juego_raw.printSchema()
juego_raw.show(3)

juego = juego_raw.select(
    F.col("juego_id").cast("int"),
    F.col("ano").alias("nombre_juego"),
    F.col("temporada").cast("int").alias("anio"),
    F.col("ciudad").alias("temporada"))

print(f"Filas del DataFrame juego: {juego.count()}  |  Años: "
      f"{juego.agg(F.min('anio')).first()[0]} a {juego.agg(F.max('anio')).first()[0]}\n")
juego.orderBy("anio").show(5)
juego.groupBy("temporada").count().show()
juego.printSchema()

Esquema original (inferido del JSON):
root
 |-- ano: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- juego_id: long (nullable = true)
 |-- temporada: long (nullable = true)

+-----------+------+--------+---------+
|        ano|ciudad|juego_id|temporada|
+-----------+------+--------+---------+
|1896 Verano|Verano|       1|     1896|
|1900 Verano|Verano|       2|     1900|
|1904 Verano|Verano|       3|     1904|
+-----------+------+--------+---------+
only showing top 3 rows
Filas del DataFrame juego: 51  |  Años: 1896 a 2016

+--------+------------+----+---------+
|juego_id|nombre_juego|anio|temporada|
+--------+------------+----+---------+
|       1| 1896 Verano|1896|   Verano|
|       2| 1900 Verano|1900|   Verano|
|       3| 1904 Verano|1904|   Verano|
|       4| 1906 Verano|1906|   Verano|
|       5| 1908 Verano|1908|   Verano|
+--------+------------+----+---------+
only showing top 5 rows
+---------+-----+
|temporada|count|
+---------+-----+
|   Verano|   29|
| 

✅ **Resultado**
- El DataFrame `juego` tiene **51 ediciones**, desde **1896 hasta 2016**: 29 de verano y 22 de invierno. Incluye los Juegos Intercalados de **1906** (Atenas).
- `multiLine=True` permitió leer el arreglo JSON completo. El esquema original mostraba los nombres engañosos (`ano`, `ciudad`, `temporada`), que corregimos a **`nombre_juego`**, **`anio`** y **`temporada`** (Verano/Invierno).
- `juego_id` y `anio` pasaron de **long a integer**, el mismo tipo que `juego_id` en `resultado`, para el join.

### 4.5 DataFrame integrado: deportista + resultado + equipo + evento + juego

🎯 **Qué haremos**
- Partir de `resultado` (tabla puente) y unirla con:
  - `deportista` por `deportista_id`,
  - `equipo` por `equipo_id`,
  - `evento` por `evento_id`,
  - `juego` por `juego_id`.
- Ordenar las columnas de forma lógica, guardar el resultado en **caché** y revisar el **plan físico** que eligió Spark.

**Decisiones técnicas (optimización)**
1. **`left join` desde `resultado`:** conservamos **todas** las participaciones (271.116), aunque falte el dato en alguna tabla (por ejemplo, los 9 `evento_id` nulos). Un `inner join` las habría eliminado en silencio.
2. **`broadcast()` en `equipo`, `evento` y `juego`:** son tablas pequeñas (1.184, 765 y 51 filas). Spark envía **una copia completa a cada ejecutor** y hace el join localmente (**BroadcastHashJoin**), sin redistribuir (*shuffle*) las 271 mil filas de resultados. `deportista` (135 mil filas) es grande, así que se une con un join normal (**SortMergeJoin**).
3. **`cache()`:** el DataFrame integrado se usará en todas las secciones siguientes. Al guardarlo en memoria evitamos releer los archivos y repetir los 4 joins en cada acción.

In [16]:
# 4.5 DataFrame integrado
import io, contextlib

integrado = (resultado
    .join(deportista,          on="deportista_id", how="left")
    .join(F.broadcast(equipo), on="equipo_id",     how="left")
    .join(F.broadcast(evento), on="evento_id",     how="left")
    .join(F.broadcast(juego),  on="juego_id",      how="left")
    .select("resultado_id", "deportista_id", "nombre", "genero", "edad", "altura", "peso",
            "equipo_id", "equipo", "sigla", "evento_id", "evento", "deporte_id",
            "juego_id", "nombre_juego", "anio", "temporada", "medalla")
    .cache())

n_int = integrado.count()          # primera acción: ejecuta los joins y llena la caché
print(f"Filas resultado : {resultado.count():,}")
print(f"Filas integrado : {n_int:,}   (¿se conservaron todas?: {n_int == resultado.count()})")
print(f"Columnas        : {len(integrado.columns)}")
print(f"¿Está en caché? : {integrado.is_cached}\n")

# Estrategias de join elegidas por Spark (extraídas del plan físico)
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    integrado.explain()
joins = [l.strip().lstrip(":+- *()0123456789") for l in buffer.getvalue().splitlines() if "Join" in l]
print("Joins en el plan físico:")
for j in joins:
    print("  ", j[:90])

print("\nParticipaciones sin dato tras el join (nulos):")
for c in ["nombre", "equipo", "evento", "nombre_juego"]:
    print(f"  {c:13s}: {integrado.filter(F.col(c).isNull()).count()}")

integrado.select("nombre", "equipo", "sigla", "evento", "nombre_juego", "temporada", "medalla").show(5, truncate=False)

Filas resultado : 271,116
Filas integrado : 271,116   (¿se conservaron todas?: True)
Columnas        : 18
¿Está en caché? : True

Joins en el plan físico:
   BroadcastHashJoin [juego_id#172], [juego_id#279], LeftOuter, BuildRight, false
   BroadcastHashJoin [evento_id#196], [evento_id#96], LeftOuter, BuildRight, false
   BroadcastHashJoin [equipo_id#6], [equipo_id#126], LeftOuter, BuildRight, false
   SortMergeJoin [deportista_id#171], [deportista_id#0], LeftOuter
   BroadcastHashJoin [juego_id#172], [juego_id#279], LeftOuter, BuildRight, false
   BroadcastHashJoin [evento_id#196], [evento_id#96], LeftOuter, BuildRight, false
   BroadcastHashJoin [equipo_id#6], [equipo_id#126], LeftOuter, BuildRight, false
   SortMergeJoin [deportista_id#171], [deportista_id#0], LeftOuter

Participaciones sin dato tras el join (nulos):
  nombre       : 0
  equipo       : 200
  evento       : 9
  nombre_juego : 0
+---------------------------+-------+-----+--------------------------------------+---------

✅ **Resultado**
- El DataFrame `integrado` tiene **271.116 filas y 18 columnas**: exactamente las mismas filas que `resultado` (`True`). El `left join` no perdió ni duplicó ninguna participación.
- Está en **caché** (`is_cached = True`). El primer `count()` ejecutó los joins y guardó el resultado en memoria para las secciones siguientes.
- **Plan físico** (cada join aparece dos veces porque Spark muestra el plan final y el inicial de la ejecución adaptativa):

| Join | Estrategia | Por qué |
|---|---|---|
| `resultado` ↔ `deportista` | **SortMergeJoin** | ambas tablas son grandes (271 mil y 135 mil filas): se redistribuyen y ordenan por `deportista_id` |
| ↔ `equipo` | **BroadcastHashJoin** | 1.184 filas: se copia a cada ejecutor, sin *shuffle* |
| ↔ `evento` | **BroadcastHashJoin** | 765 filas |
| ↔ `juego` | **BroadcastHashJoin** | 51 filas |

- **Calidad de la integración:**
  - Todas las participaciones encontraron a su deportista (0 nulos en `nombre`) y su juego (0 nulos en `nombre_juego`).
  - **9** no tienen evento: son los 9 `evento_id = "#N/A"` originales.
  - **200** no tienen equipo: el `equipo_id` de esos deportistas no existe en `equipo.csv`. Gracias al `left join` esas filas se conservan con el resto de su información.

**Cierre de la sección 4:** creamos los DataFrames `evento` (765), `equipo` (1.184), `resultado` (271.116) y `juego` (51), cada uno con tipos correctos y nombres homogéneos, y los integramos con `deportista` en un solo DataFrame optimizado con **broadcast** y **cache**.

## 5. Paralelismo (ID 3.4 · 9 pts)

### 5.1 Reparticionar el DataFrame integrado en 5 particiones

🎯 **Qué haremos**
- Ver cuántas particiones tiene hoy el DataFrame `integrado`.
- Aplicar **`repartition(5)`** y verificar que queden **5 particiones**.
- Contar las filas de **cada partición** con `spark_partition_id()` para comprobar que la carga quedó **equilibrada**.

**¿Qué es una partición y por qué importa?** Spark divide los datos en particiones, y cada partición la procesa **una tarea en un núcleo**. Con muy pocas particiones se desaprovechan núcleos; con demasiadas, el costo de coordinar tareas pequeñas supera el beneficio.

**Decisiones técnicas**
- **`repartition(n)` vs. `coalesce(n)`:** `repartition` hace un *shuffle* completo y reparte las filas de forma **pareja** (*round-robin*). `coalesce` solo junta particiones existentes sin *shuffle*: es más barato, pero puede dejar particiones desbalanceadas y no sirve para **aumentar** su número. Como la pauta pide un número exacto y queremos equilibrio, usamos `repartition`.
- **Caché:** guardamos en memoria la versión reparticionada, que es la que usaremos en las secciones siguientes, y liberamos (`unpersist`) la anterior para no duplicar memoria.

In [17]:
# 5.1 repartition(5) del DataFrame integrado
particiones_antes = integrado.rdd.getNumPartitions()

integrado_anterior = integrado
integrado = integrado.repartition(5).cache()
n = integrado.count()                      # acción: ejecuta el shuffle y llena la caché
integrado_anterior.unpersist()             # liberamos la versión anterior de la memoria

particiones_despues = integrado.rdd.getNumPartitions()
print(f"Particiones ANTES de repartition  : {particiones_antes}")
print(f"Particiones DESPUÉS de repartition: {particiones_despues}")
print(f"Filas totales (sin cambios)       : {n:,}\n")

print("Filas por partición después de repartition(5):")
(integrado.groupBy(F.spark_partition_id().alias("particion"))
          .count()
          .orderBy("particion")
          .show())

Particiones ANTES de repartition  : 8
Particiones DESPUÉS de repartition: 5
Filas totales (sin cambios)       : 271,116

Filas por partición después de repartition(5):
+---------+-----+
|particion|count|
+---------+-----+
|        0|54222|
|        1|54224|
|        2|54224|
|        3|54222|
|        4|54224|
+---------+-----+



✅ **Resultado**

| | Particiones | Filas |
|---|---:|---:|
| Antes de `repartition` | **8** | 271.116 |
| Después de `repartition(5)` | **5** | 271.116 |

- **¿Por qué había 8 antes?** El join con `deportista` (SortMergeJoin) redistribuye los datos en `spark.sql.shuffle.partitions` particiones, que configuramos en **8** en la sección 1 (una por núcleo). Esto muestra cómo una configuración de la sesión determina el paralelismo de las operaciones posteriores.
- **Después:** **5 particiones**, como pide la pauta, sin perder filas (271.116).
- **Balance:** cada partición quedó con **54.222 o 54.224 filas**, una diferencia de solo 2 filas (0,004 %). Ninguna tarea tendrá que esperar a otra mucho más pesada (*data skew*). Ese es el beneficio del reparto *round-robin* de `repartition`.
- La versión reparticionada quedó en **caché** y la anterior se liberó con `unpersist()`.

**Cierre de la sección 5:** pasamos de 8 a 5 particiones equilibradas con `repartition(5)`. Para tablas pequeñas, menos particiones reducen el costo de coordinar tareas. En un clúster real, el número de particiones se ajusta según los núcleos disponibles y el tamaño de los datos (idealmente entre 100 y 200 MB por partición).

## 6. Inspección del DataFrame integrado (ID 3.1 · 6 pts)

### 6.1 Filas, tipos de datos y esquema

🎯 **Qué haremos**
- **`count()`**: número de filas. Es una **acción**: obliga a Spark a ejecutar el plan, aunque aquí lee desde la caché.
- **`dtypes`**: lista de pares `(columna, tipo)`. Es un **atributo**, así que no ejecuta ningún trabajo: Spark ya conoce el esquema.
- **`printSchema()`**: muestra el esquema en forma de árbol, incluyendo si cada columna **admite nulos** (`nullable`).

In [18]:
# 6.1 Inspección del DataFrame integrado
print(f"Cantidad de filas    : {integrado.count():,}")
print(f"Cantidad de columnas : {len(integrado.columns)}\n")

print("Tipos de datos (dtypes):")
for columna, tipo in integrado.dtypes:
    print(f"   {columna:14s} {tipo}")

print("\nEsquema (printSchema):")
integrado.printSchema()

Cantidad de filas    : 271,116
Cantidad de columnas : 18

Tipos de datos (dtypes):
   resultado_id   int
   deportista_id  int
   nombre         string
   genero         int
   edad           int
   altura         double
   peso           double
   equipo_id      int
   equipo         string
   sigla          string
   evento_id      int
   evento         string
   deporte_id     int
   juego_id       int
   nombre_juego   string
   anio           int
   temporada      string
   medalla        string

Esquema (printSchema):
root
 |-- resultado_id: integer (nullable = true)
 |-- deportista_id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)
 |-- equipo: string (nullable = true)
 |-- sigla: string (nullable = true)
 |-- evento_id: integer (nullable = true)
 |-- evento: string (nullab

✅ **Resultado**
- El DataFrame integrado tiene **271.116 filas** y **18 columnas**.
- **Tipos de datos:** 10 columnas `int` (ids, `genero`, `edad`, `anio`), 2 `double` (`altura`, `peso`) y 6 `string` (nombres, sigla, temporada y medalla). Todos son los tipos correctos para su contenido. Ningún número quedó como texto, lo que permite operar y agregar directamente.
- **Esquema:** todas las columnas son `nullable = true`. Esto es coherente con lo que encontramos: hay nulos reales en `edad`, `altura`, `peso`, `evento` (9), `equipo` (200) y `medalla` (sin medalla).
- Diferencia clave: `count()` es una **acción** (lanza un *job*), mientras que `dtypes` y `printSchema()` solo leen los **metadatos** del esquema, sin procesar datos.

**Cierre de la sección 6:** el DataFrame integrado está completo, bien tipado y listo para calcular nuevas columnas y agregaciones.

## 7. Columnas calculadas (ID 3.3 · 12 pts)

### 7.1 `IMC` y `Descripcion_sexo`

🎯 **Qué haremos**
- **`IMC`** (Índice de Masa Corporal) = `peso / (altura/100)²`, con la altura convertida de centímetros a metros y el resultado redondeado a 2 decimales.
- **`Descripcion_sexo`**: traduce el código `genero` a texto con `when / otherwise` (1 = Hombre, 2 = Mujer).
- Mostrar ambas columnas en el DataFrame integrado y verificar el resultado.

**Decisiones técnicas**
- Usamos **funciones nativas de Spark** (`F.col`, `F.pow`, `F.round`, `F.when`) en lugar de una UDF de Python. Spark las ejecuta **dentro de la JVM**, optimizadas por Catalyst, sin enviar cada fila a Python, lo que es mucho más rápido.
- **Nulos:** si falta el peso **o** la altura, el IMC queda **nulo**. Gracias a haber tratado los ceros como nulos, evitamos un IMC de 0 o una división por cero.
- `otherwise("Sin dato")` cubre cualquier código de género distinto de 1 o 2, para que no aparezcan nulos sin explicación.

In [19]:
# 7.1 Columnas calculadas: IMC y Descripcion_sexo
integrado = (integrado
    .withColumn("IMC", F.round(F.col("peso") / F.pow(F.col("altura") / 100, 2), 2))
    .withColumn("Descripcion_sexo",
                F.when(F.col("genero") == 1, "Hombre")
                 .when(F.col("genero") == 2, "Mujer")
                 .otherwise("Sin dato")))

print("Ejemplos (incluye casos con datos faltantes):")
(integrado.filter(F.col("deportista_id").isin(1, 2, 3, 5, 8))
          .select("nombre", "genero", "Descripcion_sexo", "altura", "peso", "IMC")
          .dropDuplicates(["nombre"]).orderBy("nombre")
          .show(truncate=False))

con_imc = integrado.filter(F.col("IMC").isNotNull()).count()
print(f"Filas con IMC calculado : {con_imc:,}")
print(f"Filas sin IMC (falta peso o altura): {integrado.count() - con_imc:,}\n")

print("Resumen del IMC:")
integrado.select(F.min("IMC").alias("IMC_min"), F.round(F.avg("IMC"), 2).alias("IMC_promedio"),
                 F.max("IMC").alias("IMC_max")).show()

print("Verificación de Descripcion_sexo:")
integrado.groupBy("genero", "Descripcion_sexo").count().orderBy("genero").show()

Ejemplos (incluye casos con datos faltantes):
+-----------------------------+------+----------------+------+----+-----+
|nombre                       |genero|Descripcion_sexo|altura|peso|IMC  |
+-----------------------------+------+----------------+------+----+-----+
|A Dijiang                    |1     |Hombre          |180.0 |80.0|24.69|
|A Lamusi                     |1     |Hombre          |170.0 |60.0|20.76|
|Christine Jacoba Aaftink     |2     |Mujer           |185.0 |82.0|23.96|
|Cornelia Cor Aalten Strannood|2     |Mujer           |168.0 |NULL|NULL |
|Gunnar Nielsen Aaby          |1     |Hombre          |NULL  |NULL|NULL |
+-----------------------------+------+----------------+------+----+-----+

Filas con IMC calculado : 206,853
Filas sin IMC (falta peso o altura): 64,263

Resumen del IMC:
+-------+------------+-------+
|IMC_min|IMC_promedio|IMC_max|
+-------+------------+-------+
|   8.36|       22.78|   63.9|
+-------+------------+-------+

Verificación de Descripcion_sexo:
+

✅ **Resultado**
- **IMC** se calculó en **206.853** participaciones (76,3 %). En las **64.263** restantes (23,7 %) falta el peso o la altura, y el IMC queda nulo en vez de un valor falso. Por ejemplo, *Cornelia* tiene altura pero no peso, y *Gunnar* no tiene ninguno.
- Verificación manual: *A Dijiang* (180 cm, 80 kg) → 80 / 1,80² = **24,69**. Coincide.
- **IMC promedio: 22,78**, dentro del rango "normal" (18,5–24,9), lo esperable en deportistas de alto rendimiento.
- **Extremos:**
  - El **máximo (63,9)** corresponde a *Ricardo Blas Jr.* (183 cm, 214 kg), judoca de peso pesado: un valor real.
  - El **mínimo (8,36)** corresponde a un registro de 183 cm y 28 kg, físicamente imposible para un adulto. Es probablemente un **error de captura** del dataset. Lo dejamos visible porque el objetivo del taller no es depurar valores atípicos.
- **Descripcion_sexo**: el código 1 se tradujo a **Hombre** (196.594 participaciones) y el 2 a **Mujer** (74.522). No apareció ningún "Sin dato", así que la variable `genero` está completa.
- Nota: estas cifras son por **participación**. Un deportista que compitió en varios eventos aparece varias veces en el DataFrame integrado.

**Cierre de la sección 7:** el DataFrame integrado ahora tiene **20 columnas**, incluidas `IMC` y `Descripcion_sexo`, calculadas con funciones nativas de Spark, sin UDFs.

## 8. Agregaciones (ID 3.3 · 21 pts)

Cada agregación se resuelve de **dos formas equivalentes**:
- **API de DataFrames** (`groupBy`, `agg`, `pivot`), y
- **Spark SQL** (`spark.sql(...)`) sobre una **vista temporal** del DataFrame integrado.

Ambas pasan por el mismo optimizador (**Catalyst**) y generan planes equivalentes, por lo que el rendimiento es el mismo. Que los resultados coincidan sirve como **validación cruzada**.

### 8.1 Cantidad de medallas de oro, plata y bronce por equipo

🎯 **Qué haremos**
- Filtrar las participaciones **con medalla** y agruparlas por `equipo`.
- Con la API: usar **`pivot`** para convertir los valores de `medalla` (Gold/Silver/Bronze) en **columnas**, y sumar un `Total`.
- Con SQL: lograr lo mismo con `SUM(CASE WHEN ...)`, la forma clásica de "pivotear" en SQL.
- Ordenar de mayor a menor total y comparar ambos resultados.

**Decisiones técnicas**
- Al `pivot` le pasamos **la lista explícita** `["Gold", "Silver", "Bronze"]`. Así Spark no tiene que recorrer los datos para descubrir los valores distintos, y además fijamos el orden de las columnas.
- `na.fill(0)`: si un equipo no tiene, por ejemplo, bronces, el pivot deja `NULL`. Lo reemplazamos por 0 para poder sumar el total.
- **Criterio de conteo:** contamos **medallas por participación**. En deportes de equipo (por ejemplo, fútbol) cada integrante recibe su medalla, así que un oro en fútbol suma varias medallas. Es el criterio que permite los datos, que tienen una fila por deportista.

In [20]:
# 8.1 Medallas de oro, plata y bronce por equipo
from pyspark.sql import DataFrame

# --- API de DataFrames ---
medallas_equipo = (integrado
    .filter(F.col("medalla").isNotNull())
    .groupBy("equipo")
    .pivot("medalla", ["Gold", "Silver", "Bronze"])
    .count()
    .na.fill(0)
    .withColumnRenamed("Gold", "Oro").withColumnRenamed("Silver", "Plata").withColumnRenamed("Bronze", "Bronce")
    .withColumn("Total", F.col("Oro") + F.col("Plata") + F.col("Bronce"))
    .orderBy(F.desc("Total"), "equipo"))

print(f"Equipos con al menos una medalla: {medallas_equipo.count()}")
print("Top 10 (API de DataFrames):")
medallas_equipo.show(10)

# --- Spark SQL ---
integrado.createOrReplaceTempView("olimpicos")
medallas_equipo_sql = spark.sql("""
    SELECT equipo,
           SUM(CASE WHEN medalla = 'Gold'   THEN 1 ELSE 0 END) AS Oro,
           SUM(CASE WHEN medalla = 'Silver' THEN 1 ELSE 0 END) AS Plata,
           SUM(CASE WHEN medalla = 'Bronze' THEN 1 ELSE 0 END) AS Bronce,
           COUNT(medalla)                                      AS Total
    FROM olimpicos
    WHERE medalla IS NOT NULL
    GROUP BY equipo
    ORDER BY Total DESC, equipo
""")
print("Top 10 (Spark SQL):")
medallas_equipo_sql.show(10)

# --- Validación cruzada ---
print("¿API y SQL dan exactamente el mismo resultado?:",
      medallas_equipo.exceptAll(medallas_equipo_sql).count() == 0
      and medallas_equipo_sql.exceptAll(medallas_equipo).count() == 0)
print("Totales generales:",
      medallas_equipo.agg(F.sum("Oro"), F.sum("Plata"), F.sum("Bronce"), F.sum("Total")).first())

Equipos con al menos una medalla: 505
Top 10 (API de DataFrames):
+-------------+----+-----+------+-----+
|       equipo| Oro|Plata|Bronce|Total|
+-------------+----+-----+------+-----+
|United States|2515| 1546|  1249| 5310|
| Soviet Union|1110|  766|   728| 2604|
|      Germany| 636|  603|   651| 1890|
|Great Britain| 531|  591|   578| 1700|
|       France| 452|  524|   582| 1558|
|        Italy| 536|  507|   487| 1530|
|       Sweden| 450|  479|   510| 1439|
|    Australia| 343|  453|   506| 1302|
|       Canada| 422|  407|   407| 1236|
|      Hungary| 429|  329|   363| 1121|
+-------------+----+-----+------+-----+
only showing top 10 rows
Top 10 (Spark SQL):
+-------------+----+-----+------+-----+
|       equipo| Oro|Plata|Bronce|Total|
+-------------+----+-----+------+-----+
|United States|2515| 1546|  1249| 5310|
| Soviet Union|1110|  766|   728| 2604|
|      Germany| 636|  603|   651| 1890|
|Great Britain| 531|  591|   578| 1700|
|       France| 452|  524|   582| 1558|
|        

✅ **Resultado**
- **505 equipos** obtuvieron al menos una medalla. En total hay **39.783 medallas**: 13.372 de oro, 13.116 de plata y 13.295 de bronce, los mismos totales de la tabla `resultado`, así que no se perdió ninguna en los joins.
- **Estados Unidos** lidera con amplia ventaja: **5.310 medallas** (2.515 de oro), más del doble que la **Unión Soviética** (2.604), que es segunda pese a haber competido solo entre 1952 y 1988.
- El top 10 lo completan potencias europeas (Alemania, Gran Bretaña, Francia, Italia, Suecia, Hungría), además de Australia y Canadá.
- **Validación cruzada:** la versión con **API (`pivot`)** y la versión **SQL (`SUM(CASE WHEN ...)`)** dan exactamente el mismo resultado (`True`, verificado con `exceptAll` en ambos sentidos).

### 8.2 Edad por tipo de medalla: suma, promedio, máximo y mínimo

🎯 **Qué haremos**
- Filtrar las participaciones con medalla, agrupar por `medalla` y calcular con `agg()`: **`sum`**, **`avg`**, **`max`** y **`min`** de `edad`, más un `count` de edades disponibles.
- Repetir el cálculo con **Spark SQL** y comparar.
- Ordenar en el orden lógico Oro → Plata → Bronce (no alfabético).

**Decisión técnica:** las funciones de agregación de Spark **ignoran los nulos**. Los medallistas sin edad registrada no cuentan como "0 años", lo que habría bajado el promedio y el mínimo. Esa es otra ventaja de haber tratado los ceros como nulos en la sección 2.

In [21]:
# 8.2 Edad por tipo de medalla
orden_medalla = F.when(F.col("medalla") == "Gold", 1).when(F.col("medalla") == "Silver", 2).otherwise(3)

# --- API de DataFrames ---
edad_medalla = (integrado
    .filter(F.col("medalla").isNotNull())
    .groupBy("medalla")
    .agg(F.sum("edad").alias("suma_edad"),
         F.round(F.avg("edad"), 2).alias("promedio_edad"),
         F.max("edad").alias("max_edad"),
         F.min("edad").alias("min_edad"),
         F.count("edad").alias("n_con_edad"))
    .orderBy(orden_medalla))
print("API de DataFrames:")
edad_medalla.show()

# --- Spark SQL ---
edad_medalla_sql = spark.sql("""
    SELECT medalla,
           SUM(edad)           AS suma_edad,
           ROUND(AVG(edad), 2) AS promedio_edad,
           MAX(edad)           AS max_edad,
           MIN(edad)           AS min_edad,
           COUNT(edad)         AS n_con_edad
    FROM olimpicos
    WHERE medalla IS NOT NULL
    GROUP BY medalla
    ORDER BY CASE medalla WHEN 'Gold' THEN 1 WHEN 'Silver' THEN 2 ELSE 3 END
""")
print("Spark SQL:")
edad_medalla_sql.show()
print("¿API y SQL coinciden?:", edad_medalla.exceptAll(edad_medalla_sql).count() == 0)

# Medallistas extremos
print("\nMedallista más joven y más longevo:")
con_medalla = integrado.filter(F.col("medalla").isNotNull() & F.col("edad").isNotNull())
cols = ["nombre", "edad", "medalla", "evento", "nombre_juego"]
con_medalla.orderBy("edad").select(cols).show(1, truncate=False)
con_medalla.orderBy(F.desc("edad")).select(cols).show(1, truncate=False)

API de DataFrames:
+-------+---------+-------------+--------+--------+----------+
|medalla|suma_edad|promedio_edad|max_edad|min_edad|n_con_edad|
+-------+---------+-------------+--------+--------+----------+
|   Gold|   313374|         23.7|      64|      11|     13224|
| Silver|   304518|        23.75|      73|      11|     12821|
| Bronze|   308071|        23.69|      72|      10|     13006|
+-------+---------+-------------+--------+--------+----------+

Spark SQL:
+-------+---------+-------------+--------+--------+----------+
|medalla|suma_edad|promedio_edad|max_edad|min_edad|n_con_edad|
+-------+---------+-------------+--------+--------+----------+
|   Gold|   313374|         23.7|      64|      11|     13224|
| Silver|   304518|        23.75|      73|      11|     12821|
| Bronze|   308071|        23.69|      72|      10|     13006|
+-------+---------+-------------+--------+--------+----------+

¿API y SQL coinciden?: True

Medallista más joven y más longevo:
+------------------+-

✅ **Resultado**

| Medalla | Suma de edad | Promedio | Máx. | Mín. | Medallistas con edad |
|---|---:|---:|---:|---:|---:|
| Oro | 313.374 | 23,70 | 64 | 11 | 13.224 |
| Plata | 304.518 | 23,75 | 73 | 11 | 12.821 |
| Bronce | 308.071 | 23,69 | 72 | 10 | 13.006 |

- El **promedio de edad es prácticamente el mismo** en los tres tipos de medalla (≈ **23,7 años**): la edad no distingue entre oro, plata y bronce. La edad ideal para estar en el podio olímpico ronda los 24 años.
- **La suma depende de cuántas medallas hay** (oro tiene más), no de la edad. Por eso es una métrica poco informativa por sí sola, y la acompañamos del conteo.
- **Extremos:** el medallista más joven es *Dimitrios Loundras* (**10 años**, bronce en gimnasia por equipos, Atenas 1896). El más longevo es *John Copley* (**73 años**, plata en las **competencias de arte**, Londres 1948), que fueron parte oficial de los Juegos hasta ese año.
- Solo los medallistas **con edad registrada** entran en el cálculo (por ejemplo, 13.224 de 13.372 oros). Los nulos se ignoran en lugar de contarse como 0.
- **API y SQL coinciden** (`True`).

### 8.3 DataFrame `temporada`: altura por temporada (Verano / Invierno)

🎯 **Qué haremos**
- Crear el DataFrame **`temporada`** con la **suma, promedio, máximo y mínimo de la altura** por temporada (Verano/Invierno).
- Hacerlo con la API y con Spark SQL, y compararlos.
- Mostrar cuánto cambiaría el resultado si **no** elimináramos duplicados.

**Decisiones técnicas**
1. **Un deportista cuenta una sola vez por temporada.** La altura es un atributo **de la persona**, no de la participación. Sin `dropDuplicates(["deportista_id", "temporada"])`, un nadador con 8 pruebas pesaría 8 veces más que un deportista con una sola prueba y sesgaría el promedio.
2. **`suma_altura` como entero largo (`long`):** la suma supera los 15 millones de cm, y Spark la mostraría en notación científica (`1.5E7`). Convertirla a `long` la hace legible.
3. Los deportistas **sin altura** registrada se excluyen del cálculo (nulos), pero informamos cuántos son.

In [22]:
# 8.3 DataFrame temporada: altura por temporada
# --- API de DataFrames ---
temporada = (integrado
    .select("deportista_id", "temporada", "altura")
    .dropDuplicates(["deportista_id", "temporada"])      # 1 fila por deportista y temporada
    .groupBy("temporada")
    .agg(F.sum("altura").cast("long").alias("suma_altura"),
         F.round(F.avg("altura"), 2).alias("promedio_altura"),
         F.max("altura").alias("max_altura"),
         F.min("altura").alias("min_altura"),
         F.count("altura").alias("n_con_altura"),
         F.count("*").alias("n_deportistas"))
    .orderBy(F.desc("temporada")))
print("DataFrame temporada (API):")
temporada.show()

# --- Spark SQL ---
temporada_sql = spark.sql("""
    SELECT temporada,
           CAST(SUM(altura) AS BIGINT) AS suma_altura,
           ROUND(AVG(altura), 2)       AS promedio_altura,
           MAX(altura)                 AS max_altura,
           MIN(altura)                 AS min_altura,
           COUNT(altura)               AS n_con_altura,
           COUNT(*)                    AS n_deportistas
    FROM (SELECT DISTINCT deportista_id, temporada, altura FROM olimpicos)
    GROUP BY temporada
    ORDER BY temporada DESC
""")
print("DataFrame temporada (Spark SQL):")
temporada_sql.show()
print("¿API y SQL coinciden?:", temporada.exceptAll(temporada_sql).count() == 0)

# --- Comparación: ¿qué pasaría sin eliminar duplicados? ---
print("\nSin eliminar duplicados (cada participación cuenta):")
(integrado.groupBy("temporada")
          .agg(F.round(F.avg("altura"), 2).alias("promedio_altura"),
               F.count("altura").alias("n_registros"))
          .orderBy(F.desc("temporada"))
          .show())

DataFrame temporada (API):
+---------+-----------+---------------+----------+----------+------------+-------------+
|temporada|suma_altura|promedio_altura|max_altura|min_altura|n_con_altura|n_deportistas|
+---------+-----------+---------------+----------+----------+------------+-------------+
|   Verano|   15395905|         176.54|     226.0|     127.0|       87211|       116776|
| Invierno|    2545479|         175.01|     211.0|     137.0|       14545|        18958|
+---------+-----------+---------------+----------+----------+------------+-------------+

DataFrame temporada (Spark SQL):
+---------+-----------+---------------+----------+----------+------------+-------------+
|temporada|suma_altura|promedio_altura|max_altura|min_altura|n_con_altura|n_deportistas|
+---------+-----------+---------------+----------+----------+------------+-------------+
|   Verano|   15395905|         176.54|     226.0|     127.0|       87211|       116776|
| Invierno|    2545479|         175.01|     211.0

### 8.4 DataFrame `sexo`: edad por sexo

🎯 **Qué haremos**
- Crear el DataFrame **`sexo`** con la **suma, promedio, máximo y mínimo de la edad** por sexo, usando la columna calculada `Descripcion_sexo` (sección 7).
- Hacerlo con la API y con Spark SQL, y compararlos.
- Mostrar la diferencia con el cálculo por participación.

**Decisión técnica:** igual que en 8.3, **cada deportista cuenta una vez** (`dropDuplicates(["deportista_id"])`). Así el resultado describe a las **personas**, no a sus participaciones, y es coherente con el DataFrame `deportista` (135.571 personas).

In [24]:
# 8.4 DataFrame sexo: edad por sexo
# --- API de DataFrames ---
sexo = (integrado
    .select("deportista_id", "Descripcion_sexo", "edad")
    .dropDuplicates(["deportista_id"])                  # 1 fila por deportista
    .groupBy("Descripcion_sexo")
    .agg(F.sum("edad").cast("long").alias("suma_edad"),
         F.round(F.avg("edad"), 2).alias("promedio_edad"),
         F.max("edad").alias("max_edad"),
         F.min("edad").alias("min_edad"),
         F.count("edad").alias("n_con_edad"),
         F.count("*").alias("n_deportistas"))
    .orderBy("Descripcion_sexo"))
print("DataFrame sexo (API):")
sexo.show()

# --- Spark SQL ---
integrado.createOrReplaceTempView("olimpicos")   # refresca la vista con la versión actual del DataFrame
sexo_sql = spark.sql("""
    SELECT Descripcion_sexo,
           CAST(SUM(edad) AS BIGINT) AS suma_edad,
           ROUND(AVG(edad), 2)       AS promedio_edad,
           MAX(edad)                 AS max_edad,
           MIN(edad)                 AS min_edad,
           COUNT(edad)               AS n_con_edad,
           COUNT(*)                  AS n_deportistas
    FROM (SELECT DISTINCT deportista_id, Descripcion_sexo, edad FROM olimpicos)
    GROUP BY Descripcion_sexo
    ORDER BY Descripcion_sexo
""")
print("DataFrame sexo (Spark SQL):")
sexo_sql.show()
print("¿API y SQL coinciden?:", sorted(sexo.collect()) == sorted(sexo_sql.collect()))

# --- Comparación por participación ---
print("\nSin eliminar duplicados (cada participación cuenta):")
(integrado.groupBy("Descripcion_sexo")
          .agg(F.round(F.avg("edad"), 2).alias("promedio_edad"), F.count("edad").alias("n_registros"))
          .orderBy("Descripcion_sexo").show())

DataFrame sexo (API):
+----------------+---------+-------------+--------+--------+----------+-------------+
|Descripcion_sexo|suma_edad|promedio_edad|max_edad|min_edad|n_con_edad|n_deportistas|
+----------------+---------+-------------+--------+--------+----------+-------------+
|          Hombre|  2385884|        24.98|      97|      10|     95506|       101590|
|           Mujer|   772868|        22.94|      74|      11|     33697|        33981|
+----------------+---------+-------------+--------+--------+----------+-------------+

DataFrame sexo (Spark SQL):
+----------------+---------+-------------+--------+--------+----------+-------------+
|Descripcion_sexo|suma_edad|promedio_edad|max_edad|min_edad|n_con_edad|n_deportistas|
+----------------+---------+-------------+--------+--------+----------+-------------+
|          Hombre|  2385884|        24.98|      97|      10|     95506|       101590|
|           Mujer|   772868|        22.94|      74|      11|     33697|        33981|
+--

✅ **Resultado: DataFrame `sexo`**

| Sexo | Suma edad | Promedio | Máx. | Mín. | Con edad | Deportistas |
|---|---:|---:|---:|---:|---:|---:|
| Hombre | 2.385.884 | **24,98** | 97 | 10 | 95.506 | 101.590 |
| Mujer | 772.868 | **22,94** | 74 | 11 | 33.697 | 33.981 |

- Los **hombres son en promedio 2 años mayores** que las mujeres (24,98 vs. 22,94). Las disciplinas con mayor presencia femenina (gimnasia, natación, patinaje artístico) se caracterizan por atletas muy jóvenes.
- **Extremos:** el hombre de **97 años** y la mujer de **74** pertenecen a las antiguas **competencias de arte**. Los mínimos (10 y 11 años) corresponden a gimnastas y nadadores de los primeros Juegos.
- La suma de hombres triplica la de mujeres porque hay **3 veces más** deportistas hombres (101.590 vs. 33.981), no por la edad.
- **Por persona vs. por participación:** contando cada participación, los promedios bajan a 24,43 y 21,86. Los deportistas más jóvenes tienden a competir en **más pruebas** y pesarían varias veces en el cálculo. Por eso usamos una fila por deportista.
- **API y SQL coinciden** (`True`).

---

**Cierre de la sección 8**

| Agregación | Resultado principal | API = SQL |
|---|---|:---:|
| 8.1 Medallas por equipo | 505 equipos con medalla; EE.UU. lidera con 5.310 (2.515 oros) | ✅ |
| 8.2 Edad por medalla | promedio ≈ 23,7 años en oro, plata y bronce | ✅ |
| 8.3 DataFrame `temporada` (altura) | verano 176,54 cm vs. invierno 175,01 cm | ✅ |
| 8.4 DataFrame `sexo` (edad) | hombres 24,98 vs. mujeres 22,94 años | ✅ |

Las cuatro agregaciones se resolvieron con la **API de DataFrames** y con **Spark SQL** sobre la vista temporal `olimpicos`, con resultados idénticos. Además, en las agregaciones de **atributos personales** (altura, edad) cada deportista se contó **una sola vez**, para no sobrerrepresentar a quienes compiten en muchas pruebas.